# Study 893 — Vol-Target 60/40 🌡️

**Put a thermostat on the balanced book: hold *less* when it's stormy, *more* when it's calm — does
the classic 60/40 come out better?**

The 60/40 (60% stocks, 40% bonds) is the sensible default ([Study 97](../../97-balancing-act/) grades
it Real & Investable). Its risk, though, is *not* constant — it doubles in a crisis. So we bolt on the
inverse-volatility overlay this desk certified on equities ([Study 16, Storm-Shy](../../16-storm-shy/)):
scale the *whole book's* exposure so realized **portfolio** volatility stays near a constant target.
Real tape SPY/IEF, cash = BIL, 2007-05-31 → 2026-06-30.

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `a874e54fa109`); the live cells run
the fast synthetic control. Short history: BIL lists 2007 — a single-cycle, GFC-anchored window.*


In [1]:
R = {'alpha_ann': 2.31,
 'avg_lev': 1.36,
 'beta': 0.931,
 'boot_hi': 0.323,
 'boot_lo': -0.086,
 'boot_point': 0.126,
 'boot_win': 87.8,
 'cost0': 0.107,
 'cost1': 0.098,
 'cost10': 0.023,
 'cost10_dd': -25.3,
 'cost2': 0.09,
 'cost5': 0.065,
 'crash08_s': -14.9,
 'crash08_v': -13.2,
 'crash22_s': -17.0,
 'crash22_v': -14.1,
 'diff_t_nw': 1.63,
 'end': '2026-06-30',
 'era_e_dd_s': -29.6,
 'era_e_dd_v': -25.6,
 'era_e_gain': 0.217,
 'era_e_n': 1891,
 'era_e_t': 1.86,
 'era_l_dd_s': -21.4,
 'era_l_dd_v': -16.4,
 'era_l_gain': 0.058,
 'era_l_n': 2868,
 'era_l_t': 0.93,
 'fp': 'a874e54fa109',
 'frac_capped': 18,
 'frac_lev': 74,
 'n_days': 4801,
 'n_seeds': 30,
 'null_fire': 2,
 'null_t_mean': 0.2,
 'plan_fire': 23,
 'plan_t_mean': 3.09,
 'sharpe_gain': 0.126,
 'start': '2007-05-31',
 'static_cagr': 8.43,
 'static_dd': -29.6,
 'static_sharpe': 0.682,
 'static_vol': 10.68,
 't_alpha': 1.92,
 'target_vol': 10.67,
 'turnover': 9.4,
 'vt_cagr': 10.33,
 'vt_dd': -24.2,
 'vt_sharpe': 0.808,
 'vt_vol': 11.26,
 'win21': 0.126,
 'win42': 0.088,
 'win63': 0.054}

## 1. The thermostat, in one line

Volatility **clusters** — a wild week is usually followed by another wild week, a calm one by calm. So *yesterday's* realized vol is a decent guess for *today's*. The rule uses only the past: `weight = target_vol / recent_portfolio_vol`, capped at 2×. When the 60/40 gets stormy you automatically hold less; when it's sleepy you hold a bit more (financed at cash). Same average risk, just **re-timed**.

In [2]:
print(f"static 60/40   : excess Sharpe {R['static_sharpe']:.3f}, maxDD {R['static_dd']:.1f}%")
print(f"vol-targeted   : excess Sharpe {R['vt_sharpe']:.3f}, maxDD {R['vt_dd']:.1f}%")
print(f"Sharpe gain    : {R['sharpe_gain']:+.3f}  (drawdown cut by {R['static_dd']-R['vt_dd']:+.1f} pts)")

static 60/40   : excess Sharpe 0.682, maxDD -29.6%
vol-targeted   : excess Sharpe 0.808, maxDD -24.2%
Sharpe gain    : +0.126  (drawdown cut by -5.4 pts)


## 2. The honest catch — a smoother ride, but the Sharpe lift is *not* certain

The point estimates all lean the thermostat's way (Sharpe **0.68 → 0.81**, drawdown **-30% → -24%**). But when we ask *how sure are we?*, the block-bootstrap band on the Sharpe gain is **[-0.09, +0.32]** — it **straddles zero** (the thermostat wins in 88% of resamples, so ~12% of the time it loses). And the leverage-clean significance *t* is **1.92** — just under the desk's bar of 2.

In [3]:
print(f"bootstrap Sharpe gain {R['boot_point']:+.3f}  95% CI [{R['boot_lo']:+.3f}, {R['boot_hi']:+.3f}]")
print(f"  -> straddles zero; vol-target wins {R['boot_win']:.1f}% of resamples")
print(f"spanning-alpha t = {R['t_alpha']:.2f}  (< 2 -> sub-significant)")

bootstrap Sharpe gain +0.126  95% CI [-0.086, +0.323]
  -> straddles zero; vol-target wins 87.8% of resamples
spanning-alpha t = 1.92  (< 2 -> sub-significant)


## 3. What *is* rock-solid: the drawdown, and it holds in every crash

The one thing that survives every era and every cost is the **shallower drawdown**. In the two worst years for the balanced book:

- **2008:** -14.9% → **-13.2%**
- **2022** (stocks *and* bonds fell together): -17.0% → **-14.1%**

You run this overlay for the *smoother ride*, not for a guaranteed Sharpe pickup.

## 4. Is the machinery honest? A live synthetic control

We plant vol-clustering in a seeded toy 60/40, and a flat-vol twin where there's nothing to forecast. The detector must light up on the first and stay silent on the second. No network.

In [4]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from vt6040 import data, strategy as st
planted = st.synthetic_detect(data.synthetic_prices(seed=904, n_days=6000)[0])
null    = st.synthetic_detect(data.synthetic_prices(seed=904, n_days=6000, sigma_hi=0.006)[0])
print(f"planted (clustered): Sharpe gain {planted['sharpe_gain']:+.3f}, spanning-alpha t {planted['t_alpha']:+.2f}  (should light up)")
print(f"null    (flat vol) : Sharpe gain {null['sharpe_gain']:+.3f}, spanning-alpha t {null['t_alpha']:+.2f}  (should be ~0)")

planted (clustered): Sharpe gain +0.412, spanning-alpha t +4.74  (should light up)
null    (flat vol) : Sharpe gain -0.049, spanning-alpha t -0.62  (should be ~0)


## 5. The verdict

- **Signal — Weak.** Every point estimate favours the thermostat and the **drawdown reduction is real and robust** — but the risk-adjusted *improvement* doesn't clear the bar (spanning-alpha *t* 1.92, bootstrap CI straddles zero, and the edge fades from +0.22 pre-2015 to +0.06 after).
- **Tradability — Fragile.** You can run it cheaply on the two most liquid ETFs alive, and the shallower drawdown is bankable — but the thin, decaying Sharpe edge leans on leverage (avg 1.36×) and a borrow spread eats most of it. A risk overlay you deploy for the ride, not a certified free lunch.